# 02 — Splits fruit-disjoint + baselines de regresión

## Proyecto
**Kaki_DeepHS_StorageDays**

## Objetivo
Evaluar si los espectros HSI permiten estimar los **días de almacenamiento** en frutos físicos nunca vistos durante el entrenamiento.

Este notebook parte de los espectros agregados y aprobados del notebook 01:

- `VIS`: 199 observaciones fruto×día, 224 bandas.
- `VIS_COR`: 199 observaciones fruto×día, 249 bandas.
- 40 frutos físicos.
- Días disponibles: `0, 1, 2, 3, 4, 6, 7, 8`.
- `front/back` ya fueron agregados mediante mediana.
- MASK V2 aprobada.

## Regla de validación

Todos los días de un fruto pertenecen al mismo fold:

\[
fruit_i \in train \quad \text{o} \quad fruit_i \in test
\]

Nunca ambos.

La evaluación principal usa **5 folds externos congelados**, cada uno con 8 frutos físicos. Dentro de cada fold externo se realiza selección de preprocesamiento e hiperparámetros con **4 folds internos fruit-disjoint**.

## Modelos baseline

1. `DummyMedian` — referencia nula.
2. `Ridge`
3. `PLSR`
4. `SVR-RBF`

Los sensores `VIS` y `VIS_COR` se modelan **por separado**.

## Preprocesamientos candidatos

Para Ridge, PLSR y SVR:

- `raw`
- `SNV`
- `SG1`: primera derivada Savitzky–Golay

La selección ocurre **solo dentro de los datos de entrenamiento del fold externo**.

## Métricas

- MAE fruto-día
- RMSE
- R²
- Spearman
- Accuracy ±1 día
- **MAE fruit-balanced**: cada fruto pesa lo mismo
- **MAE day-balanced**: cada día pesa lo mismo
- Bias medio

> El objetivo de este notebook es evaluar generalización a frutos nuevos **dentro de la misma campaña experimental**. El diseño no elimina la confusión `storage_day ↔ acquisition_session`.


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import itertools
import json
import math
import os
import platform
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import savgol_filter
from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "02_kaki_fruit_disjoint_splits_baselines"
RUN_SIGNATURE = "baseline_v1_maskV2"

ROOT = PROJECT_ROOT

INTERIM = ROOT / "data/interim"
SPLITS_DIR = ROOT / "data/splits"

RESULTS = ROOT / "results/02_kaki_fruit_disjoint_splits_baselines"
FIGURES = ROOT / "figures/02_kaki_fruit_disjoint_splits_baselines"
TABLES = ROOT / "tables/02_kaki_fruit_disjoint_splits_baselines"
EXPORT_ROOT = ROOT / "exports/02_kaki_fruit_disjoint_splits_baselines"

CHECKPOINT_DIR = RESULTS / f"checkpoints_{RUN_SIGNATURE}"

for p in [
    SPLITS_DIR,
    RESULTS,
    FIGURES,
    TABLES,
    EXPORT_ROOT,
    CHECKPOINT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260818
OUTER_FOLDS = 5
INNER_FOLDS = 4

# Si Colab se desconecta, volver a ejecutar el notebook con RESUME=True
# reutiliza combinaciones ya terminadas.
RESUME = True

# Bootstrap a nivel de fruto para CI descriptivo del MAE fruit-balanced.
BOOTSTRAP_REPS = 2000

print("ROOT:", ROOT)
print("RUN_SIGNATURE:", RUN_SIGNATURE)
print("RESUME:", RESUME)


## 2. Verificar que los espectros corresponden a MASK V2

Esta es una compuerta dura. El modelado no continúa si los caches espectrales parecen provenir de la máscara antigua.


In [ ]:
cube_cache_paths = {
    "VIS": INTERIM / "kaki_cube_spectra_VIS.csv",
    "VIS_COR": INTERIM / "kaki_cube_spectra_VIS_COR.csv",
}

expected_cube_counts = {
    "VIS": 395,
    "VIS_COR": 398,
}

mask_qc_rows = []

for camera, path in cube_cache_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró el cache aprobado del notebook 01: {path}"
        )

    df = pd.read_csv(path)

    median_coverage = float(df["mask_coverage"].median())

    row = {
        "camera_type": camera,
        "n_cubes": int(len(df)),
        "median_mask_coverage": median_coverage,
        "n_mask_qc_not_ok": int((df["mask_qc_flag"] != "ok").sum()),
        "n_extraction_errors": int(df["extraction_error"].notna().sum()),
    }

    mask_qc_rows.append(row)

    assert len(df) == expected_cube_counts[camera]
    assert median_coverage < 0.87, (
        f"{camera}: mediana de máscara {median_coverage:.6f}. "
        "Parece MASK V1; detener modelado."
    )
    assert row["n_mask_qc_not_ok"] == 0
    assert row["n_extraction_errors"] == 0

mask_qc = pd.DataFrame(mask_qc_rows)
display(mask_qc)

mask_qc.to_csv(
    TABLES / "01_mask_v2_modeling_gate.csv",
    index=False
)

print("✅ MASK V2 confirmada para ambos sensores.")


## 3. Cargar espectros fruto×día×cámara


In [ ]:
spectra_paths = {
    "VIS": INTERIM / "kaki_fruit_day_spectra_VIS.csv",
    "VIS_COR": INTERIM / "kaki_fruit_day_spectra_VIS_COR.csv",
}

spectral_data = {}
wavelengths = {}

for camera, path in spectra_paths.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Falta espectro agregado del notebook 01: {path}"
        )

    df = pd.read_csv(path)

    required = [
        "fruit_id",
        "storage_days_model",
        "day_code",
        "camera_type",
        "n_sides_aggregated",
        "median_mask_coverage",
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(
            f"{camera}: faltan columnas {missing}"
        )

    wl_cols = [
        c for c in df.columns
        if c.startswith("wl_")
    ]

    if not wl_cols:
        raise RuntimeError(
            f"{camera}: no hay columnas espectrales."
        )

    df["fruit_id"] = pd.to_numeric(
        df["fruit_id"],
        errors="raise"
    ).astype(int)

    df["storage_days_model"] = pd.to_numeric(
        df["storage_days_model"],
        errors="raise"
    ).astype(int)

    assert len(df) == 199
    assert df["fruit_id"].nunique() == 40
    assert (
        df[["fruit_id", "storage_days_model"]]
        .duplicated()
        .sum()
        == 0
    )
    assert not df[wl_cols].isna().any().any()

    spectral_data[camera] = df.sort_values(
        ["fruit_id", "storage_days_model"]
    ).reset_index(drop=True)

    wavelengths[camera] = np.array(
        [
            float(c.replace("wl_", ""))
            for c in wl_cols
        ],
        dtype=float
    )

    print(
        camera,
        "| rows:", len(df),
        "| fruits:", df["fruit_id"].nunique(),
        "| bands:", len(wl_cols),
        "| range:",
        f"{wavelengths[camera].min():.3f}–{wavelengths[camera].max():.3f} nm"
    )


## 4. Confirmar que VIS y VIS_COR tienen exactamente el mismo panel longitudinal


In [ ]:
keys_vis = set(
    map(
        tuple,
        spectral_data["VIS"][
            ["fruit_id", "storage_days_model"]
        ].to_numpy()
    )
)

keys_cor = set(
    map(
        tuple,
        spectral_data["VIS_COR"][
            ["fruit_id", "storage_days_model"]
        ].to_numpy()
    )
)

assert keys_vis == keys_cor
assert len(keys_vis) == 199

print("✅ Ambos sensores contienen los mismos 199 fruit-days.")


# 5. Construir y congelar los 5 folds externos

Los folds se fijaron una sola vez usando únicamente:

- `fruit_id`;
- día terminal del fruto;
- número de observaciones longitudinales por fruto.

**No se utilizaron los valores espectrales para diseñar los folds.**

Cada fold externo contiene exactamente 8 frutos.


In [ ]:
# Estadística por fruto, idéntica para ambos sensores.
base = spectral_data["VIS"]

fruit_stats = (
    base.groupby("fruit_id")
    .agg(
        terminal_day=("storage_days_model", "max"),
        n_fruit_days=("storage_days_model", "size"),
    )
    .reset_index()
    .sort_values("fruit_id")
)

assert len(fruit_stats) == 40

display(
    fruit_stats["terminal_day"]
    .value_counts()
    .sort_index()
    .rename("n_fruits")
    .to_frame()
)

# Folds externos congelados.
# Cada fruta aparece una sola vez.
FROZEN_OUTER_FOLDS = {
    0: [2, 4, 6, 7, 16, 26, 28, 37],
    1: [13, 18, 23, 27, 31, 36, 39, 40],
    2: [1, 10, 12, 17, 19, 30, 33, 38],
    3: [9, 11, 14, 15, 21, 24, 25, 34],
    4: [3, 5, 8, 20, 22, 29, 32, 35],
}

all_frozen = [
    fruit
    for fruits in FROZEN_OUTER_FOLDS.values()
    for fruit in fruits
]

assert len(all_frozen) == 40
assert len(set(all_frozen)) == 40
assert set(all_frozen) == set(fruit_stats["fruit_id"])

outer_split_rows = []

for fold, fruits in FROZEN_OUTER_FOLDS.items():
    for fruit_id in fruits:
        row = fruit_stats[
            fruit_stats["fruit_id"] == fruit_id
        ].iloc[0]

        outer_split_rows.append({
            "fruit_id": int(fruit_id),
            "outer_fold": int(fold),
            "terminal_day": int(row["terminal_day"]),
            "n_fruit_days": int(row["n_fruit_days"]),
        })

outer_splits = (
    pd.DataFrame(outer_split_rows)
    .sort_values(["outer_fold", "fruit_id"])
    .reset_index(drop=True)
)

OUTER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_outer5_fruit_disjoint_v1.csv"
)

if OUTER_SPLIT_PATH.exists():

    existing = (
        pd.read_csv(OUTER_SPLIT_PATH)
        .sort_values(["outer_fold", "fruit_id"])
        .reset_index(drop=True)
    )

    cols = [
        "fruit_id",
        "outer_fold",
        "terminal_day",
        "n_fruit_days",
    ]

    if not existing[cols].equals(
        outer_splits[cols]
    ):
        raise RuntimeError(
            "Existe un archivo de splits con una asignación diferente. "
            "No se sobrescribirá silenciosamente."
        )

    print("✅ Split externo congelado ya existente; se reutiliza.")

else:

    outer_splits.to_csv(
        OUTER_SPLIT_PATH,
        index=False
    )

    print("✅ Split externo congelado guardado por primera vez.")

display(outer_splits.head(16))


## 6. Auditoría de balance de los folds externos


In [ ]:
outer_balance = (
    outer_splits.groupby("outer_fold")
    .agg(
        n_fruits=("fruit_id", "nunique"),
        total_fruit_days=("n_fruit_days", "sum"),
        terminal_day_mean=("terminal_day", "mean"),
        terminal_day_median=("terminal_day", "median"),
        terminal_day_min=("terminal_day", "min"),
        terminal_day_max=("terminal_day", "max"),
    )
    .reset_index()
)

display(outer_balance)

assert (outer_balance["n_fruits"] == 8).all()

terminal_hist = pd.crosstab(
    outer_splits["outer_fold"],
    outer_splits["terminal_day"]
)

display(terminal_hist)

outer_balance.to_csv(
    TABLES / "02_outer_fold_balance.csv",
    index=False
)

terminal_hist.to_csv(
    TABLES / "03_outer_fold_terminal_day_histogram.csv"
)

fig, ax = plt.subplots(figsize=(9, 4.8))

img = ax.imshow(
    terminal_hist.to_numpy(),
    aspect="auto"
)

ax.set_xticks(range(len(terminal_hist.columns)))
ax.set_xticklabels([str(int(x)) for x in terminal_hist.columns])

ax.set_yticks(range(len(terminal_hist.index)))
ax.set_yticklabels([str(int(x)) for x in terminal_hist.index])

ax.set_xlabel("Día terminal del fruto")
ax.set_ylabel("Outer fold")
ax.set_title("Balance de día terminal en los 5 folds externos")

fig.colorbar(
    img,
    ax=ax,
    label="Número de frutos"
)

fig.tight_layout()
fig.savefig(
    FIGURES / "01_outer_fold_terminal_day_balance.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# 7. Folds internos fruit-disjoint

Para cada fold externo:

- 8 frutos = test externo;
- 32 frutos = desarrollo;
- los 32 frutos de desarrollo se dividen en 4 folds internos de 8 frutos.

La búsqueda interna usa únicamente los 32 frutos de desarrollo.


In [ ]:
def make_balanced_inner_folds(
    fruit_table,
    n_splits,
    seed,
    n_iter=5000
):
    """
    Busca una partición determinista aproximadamente balanceada
    por terminal_day y número de fruit-days.

    No usa datos espectrales.
    """

    ft = (
        fruit_table
        .sort_values("fruit_id")
        .reset_index(drop=True)
        .copy()
    )

    n = len(ft)

    if n < n_splits:
        raise ValueError("Menos frutos que folds.")

    fids = ft["fruit_id"].to_numpy(dtype=int)
    terminal = ft["terminal_day"].to_numpy(dtype=int)
    nobs = ft["n_fruit_days"].to_numpy(dtype=int)

    all_terminal_days = np.array(
        sorted(fruit_stats["terminal_day"].unique()),
        dtype=int
    )

    target_hist = np.array(
        [
            np.sum(terminal == d)
            for d in all_terminal_days
        ],
        dtype=float
    ) / n_splits

    target_obs = nobs.sum() / n_splits
    target_terminal_mean = terminal.mean()

    fold_sizes = [
        n // n_splits
        + (1 if i < (n % n_splits) else 0)
        for i in range(n_splits)
    ]

    rng_local = np.random.default_rng(seed)

    best_score = np.inf
    best_assignment = None

    for _ in range(n_iter):

        permutation = rng_local.permutation(n)

        offset = 0
        candidate = []
        score = 0.0

        for fold, size in enumerate(fold_sizes):

            idx = permutation[
                offset:offset + size
            ]

            offset += size

            t = terminal[idx]
            o = nobs[idx]

            hist = np.array(
                [
                    np.sum(t == d)
                    for d in all_terminal_days
                ],
                dtype=float
            )

            score += np.sum(
                (hist - target_hist) ** 2
                /
                (target_hist + 0.5)
            )

            obs_scale = max(
                target_obs * 0.10,
                1.0
            )

            score += (
                (o.sum() - target_obs)
                /
                obs_scale
            ) ** 2

            score += (
                (t.mean() - target_terminal_mean)
                /
                1.5
            ) ** 2

            candidate.append(
                fids[idx].copy()
            )

        if score < best_score:

            best_score = score
            best_assignment = candidate

    rows = []

    for inner_fold, fruits in enumerate(
        best_assignment
    ):

        for fruit_id in fruits:

            rows.append({
                "fruit_id": int(fruit_id),
                "inner_fold": int(inner_fold),
            })

    return (
        pd.DataFrame(rows)
        .sort_values(["inner_fold", "fruit_id"])
        .reset_index(drop=True)
    )


inner_split_rows = []

for outer_fold in range(OUTER_FOLDS):

    outer_test_fruits = set(
        outer_splits.loc[
            outer_splits["outer_fold"] == outer_fold,
            "fruit_id"
        ].astype(int)
    )

    development_stats = fruit_stats[
        ~fruit_stats["fruit_id"].isin(
            outer_test_fruits
        )
    ].copy()

    inner = make_balanced_inner_folds(
        development_stats,
        n_splits=INNER_FOLDS,
        seed=RANDOM_SEED + outer_fold,
        n_iter=5000
    )

    assert len(inner) == 32
    assert inner["fruit_id"].nunique() == 32
    assert not (
        set(inner["fruit_id"])
        &
        outer_test_fruits
    )

    for _, row in inner.iterrows():
        inner_split_rows.append({
            "outer_fold": int(outer_fold),
            "fruit_id": int(row["fruit_id"]),
            "inner_fold": int(row["inner_fold"]),
        })

inner_splits = pd.DataFrame(inner_split_rows)

INNER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_inner4_within_outer5_fruit_disjoint_v1.csv"
)

if INNER_SPLIT_PATH.exists():

    existing = (
        pd.read_csv(INNER_SPLIT_PATH)
        .sort_values(
            ["outer_fold", "inner_fold", "fruit_id"]
        )
        .reset_index(drop=True)
    )

    expected = (
        inner_splits
        .sort_values(
            ["outer_fold", "inner_fold", "fruit_id"]
        )
        .reset_index(drop=True)
    )

    if not existing.equals(expected):
        raise RuntimeError(
            "El archivo de folds internos existente no coincide "
            "con la asignación reproducida."
        )

    print("✅ Folds internos existentes verificados.")

else:

    inner_splits.to_csv(
        INNER_SPLIT_PATH,
        index=False
    )

    print("✅ Folds internos guardados.")

display(
    inner_splits.groupby(
        ["outer_fold", "inner_fold"]
    )["fruit_id"]
    .nunique()
    .unstack()
)

assert (
    inner_splits.groupby(
        ["outer_fold", "inner_fold"]
    )["fruit_id"]
    .nunique()
    .eq(8)
    .all()
)


# 8. Auditoría anti-leakage

El notebook se detiene si:

- un fruto aparece en train y test del mismo fold externo;
- un fruto test aparece en los folds internos de ese outer fold;
- una observación fruto×día aparece más de una vez en OOF.


In [ ]:
leakage_rows = []

all_fruits = set(
    outer_splits["fruit_id"].astype(int)
)

for outer_fold in range(OUTER_FOLDS):

    test_fruits = set(
        outer_splits.loc[
            outer_splits["outer_fold"] == outer_fold,
            "fruit_id"
        ].astype(int)
    )

    train_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_fruits = set(
        inner_splits.loc[
            inner_splits["outer_fold"] == outer_fold,
            "fruit_id"
        ].astype(int)
    )

    leakage_rows.append({
        "outer_fold": outer_fold,
        "n_train_fruits": len(train_fruits),
        "n_test_fruits": len(test_fruits),
        "train_test_overlap": len(
            train_fruits & test_fruits
        ),
        "inner_test_overlap": len(
            inner_fruits & test_fruits
        ),
        "inner_matches_train": (
            inner_fruits == train_fruits
        ),
    })

leakage_audit = pd.DataFrame(leakage_rows)

display(leakage_audit)

assert (
    leakage_audit["train_test_overlap"] == 0
).all()

assert (
    leakage_audit["inner_test_overlap"] == 0
).all()

assert leakage_audit[
    "inner_matches_train"
].all()

leakage_audit.to_csv(
    TABLES / "04_split_leakage_audit.csv",
    index=False
)

print("✅ No hay fuga de identidad de fruto en los splits.")


# 9. Preprocesamiento espectral

Los transformadores `raw`, `SNV` y `SG1` operan fila por fila y no aprenden parámetros del test.

Para Ridge y SVR se añade `StandardScaler` dentro del pipeline, ajustado únicamente con el training correspondiente.

PLSR usa su escalado interno sobre training.


In [ ]:
PREPROCESSING_METHODS = [
    "raw",
    "snv",
    "sg1",
]

def preprocess_spectra(
    X,
    wavelength_grid,
    method
):
    X = np.asarray(
        X,
        dtype=float
    )

    if method == "raw":
        return X.copy()

    if method == "snv":

        means = np.mean(
            X,
            axis=1,
            keepdims=True
        )

        stds = np.std(
            X,
            axis=1,
            keepdims=True
        )

        stds[
            (~np.isfinite(stds))
            |
            (stds == 0)
        ] = 1.0

        return (
            X - means
        ) / stds

    if method == "sg1":

        grid = np.asarray(
            wavelength_grid,
            dtype=float
        )

        delta = float(
            np.median(
                np.diff(grid)
            )
        )

        return savgol_filter(
            X,
            window_length=11,
            polyorder=2,
            deriv=1,
            delta=delta,
            axis=1,
            mode="interp"
        )

    raise ValueError(
        f"Preprocesamiento desconocido: {method}"
    )


# Sanity check dimensional
for camera, df in spectral_data.items():

    wl_cols = [
        c for c in df.columns
        if c.startswith("wl_")
    ]

    X0 = df[wl_cols].head(3).to_numpy()

    for method in PREPROCESSING_METHODS:

        Xt = preprocess_spectra(
            X0,
            wavelengths[camera],
            method
        )

        assert Xt.shape == X0.shape
        assert np.isfinite(Xt).all()

print("✅ Preprocesamientos validados.")


## 10. Espacio de hiperparámetros


In [ ]:
PARAM_GRIDS = {

    "Ridge": {
        "alpha": [
            0.01,
            0.1,
            1.0,
            10.0,
            100.0,
            1000.0,
        ]
    },

    "PLSR": {
        "n_components": [
            2,
            4,
            6,
            8,
            10,
            12,
            15,
        ]
    },

    "SVR": {
        "C": [
            1.0,
            10.0,
            100.0,
        ],
        "epsilon": [
            0.25,
            0.5,
            1.0,
        ],
    },
}


def expand_grid(grid):

    keys = list(grid.keys())

    for values in itertools.product(
        *[
            grid[k]
            for k in keys
        ]
    ):
        yield dict(
            zip(
                keys,
                values
            )
        )


model_config_counts = []

for model_name, grid in PARAM_GRIDS.items():

    n_param = len(
        list(
            expand_grid(grid)
        )
    )

    n_total = (
        n_param
        *
        len(PREPROCESSING_METHODS)
    )

    model_config_counts.append({
        "model": model_name,
        "parameter_combinations": n_param,
        "preprocessing_options": len(PREPROCESSING_METHODS),
        "total_candidate_configs": n_total,
    })

display(
    pd.DataFrame(
        model_config_counts
    )
)


# 11. Funciones de ajuste y métricas

La métrica usada para seleccionar hiperparámetros es:

\[
\mathrm{MAE}_{fruit-balanced}
=

rac{1}{N_{fruit}}
\sum_i
\mathrm{MAE}_i
\]

Así un fruto con 8 días no pesa ocho veces más que un fruto con 1 día.


In [ ]:
def build_estimator(
    model_name,
    params
):

    if model_name == "Ridge":

        return Pipeline([
            (
                "scale",
                StandardScaler()
            ),
            (
                "model",
                Ridge(
                    alpha=float(
                        params["alpha"]
                    )
                )
            ),
        ])

    if model_name == "PLSR":

        return PLSRegression(
            n_components=int(
                params[
                    "n_components"
                ]
            ),
            scale=True,
            max_iter=1000
        )

    if model_name == "SVR":

        return Pipeline([
            (
                "scale",
                StandardScaler()
            ),
            (
                "model",
                SVR(
                    kernel="rbf",
                    C=float(
                        params["C"]
                    ),
                    epsilon=float(
                        params["epsilon"]
                    ),
                    gamma="scale",
                )
            ),
        ])

    raise ValueError(
        f"Modelo desconocido: {model_name}"
    )


def fit_predict_estimator(
    model_name,
    params,
    X_train,
    y_train,
    X_test
):

    estimator = build_estimator(
        model_name,
        params
    )

    estimator.fit(
        X_train,
        y_train
    )

    pred = np.asarray(
        estimator.predict(
            X_test
        )
    ).ravel()

    return pred, estimator


def fruit_balanced_mae(
    prediction_df
):

    tmp = prediction_df.copy()

    tmp["abs_error"] = np.abs(
        tmp["y_true"]
        -
        tmp["y_pred"]
    )

    return float(
        tmp.groupby(
            "fruit_id"
        )["abs_error"]
        .mean()
        .mean()
    )


def day_balanced_mae(
    prediction_df
):

    tmp = prediction_df.copy()

    tmp["abs_error"] = np.abs(
        tmp["y_true"]
        -
        tmp["y_pred"]
    )

    return float(
        tmp.groupby(
            "y_true"
        )["abs_error"]
        .mean()
        .mean()
    )


def metric_bundle(
    prediction_df
):

    y_true = prediction_df[
        "y_true"
    ].to_numpy(dtype=float)

    y_pred = prediction_df[
        "y_pred"
    ].to_numpy(dtype=float)

    residual = (
        y_pred - y_true
    )

    rho, _ = spearmanr(
        y_true,
        y_pred
    )

    return {

        "n_observations":
            int(len(prediction_df)),

        "n_fruits":
            int(
                prediction_df[
                    "fruit_id"
                ].nunique()
            ),

        "mae":
            float(
                mean_absolute_error(
                    y_true,
                    y_pred
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true,
                        y_pred
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y_true,
                    y_pred
                )
            ),

        "spearman_rho":
            float(rho),

        "within_1_day_accuracy":
            float(
                np.mean(
                    np.abs(
                        residual
                    )
                    <= 1.0
                )
            ),

        "fruit_balanced_mae":
            fruit_balanced_mae(
                prediction_df
            ),

        "day_balanced_mae":
            day_balanced_mae(
                prediction_df
            ),

        "bias":
            float(
                np.mean(
                    residual
                )
            ),
    }


def config_id(
    preprocessing,
    params
):

    payload = {
        "preprocessing": preprocessing,
        "params": params,
    }

    return json.dumps(
        payload,
        sort_keys=True
    )


# 12. Selección interna de configuración

Para cada combinación:

- preprocesamiento;
- hiperparámetros;
- 4 folds internos fruit-disjoint.

Se concatenan las predicciones internas y se calcula `fruit_balanced_mae`.

No se usa el test externo para elegir configuración.


In [ ]:
def inner_select_configuration(
    camera,
    model_name,
    outer_fold,
    data_df,
    wavelength_grid,
    inner_assignment
):

    wl_cols = [
        c for c in data_df.columns
        if c.startswith("wl_")
    ]

    candidate_configs = []

    for preprocessing in PREPROCESSING_METHODS:

        for params in expand_grid(
            PARAM_GRIDS[
                model_name
            ]
        ):

            candidate_configs.append(
                (
                    preprocessing,
                    params
                )
            )

    search_rows = []

    best = None

    iterator = tqdm(
        candidate_configs,
        desc=(
            f"{camera} | outer {outer_fold} | "
            f"{model_name} | inner search"
        ),
        leave=False
    )

    for preprocessing, params in iterator:

        config_key = config_id(
            preprocessing,
            params
        )

        prediction_parts = []

        failed = False
        failure_text = None

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            mask_val = (
                data_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            )

            mask_train = (
                ~mask_val
            )

            train_df = data_df[
                mask_train
            ]

            val_df = data_df[
                mask_val
            ]

            X_train_raw = train_df[
                wl_cols
            ].to_numpy(dtype=float)

            X_val_raw = val_df[
                wl_cols
            ].to_numpy(dtype=float)

            y_train = train_df[
                "storage_days_model"
            ].to_numpy(dtype=float)

            try:

                X_train = preprocess_spectra(
                    X_train_raw,
                    wavelength_grid,
                    preprocessing
                )

                X_val = preprocess_spectra(
                    X_val_raw,
                    wavelength_grid,
                    preprocessing
                )

                pred, _ = fit_predict_estimator(
                    model_name,
                    params,
                    X_train,
                    y_train,
                    X_val
                )

                prediction_parts.append(
                    pd.DataFrame({
                        "fruit_id":
                            val_df[
                                "fruit_id"
                            ].to_numpy(dtype=int),

                        "y_true":
                            val_df[
                                "storage_days_model"
                            ].to_numpy(dtype=float),

                        "y_pred":
                            pred,

                        "inner_fold":
                            int(inner_fold),
                    })
                )

            except Exception as exc:

                failed = True
                failure_text = repr(exc)
                break

        if failed:

            search_rows.append({
                "camera_type": camera,
                "outer_fold": int(outer_fold),
                "model": model_name,
                "preprocessing": preprocessing,
                "params_json": json.dumps(
                    params,
                    sort_keys=True
                ),
                "config_id": config_key,
                "inner_fruit_balanced_mae": np.inf,
                "inner_mae": np.inf,
                "failed": True,
                "failure": failure_text,
            })

            continue

        inner_pred = pd.concat(
            prediction_parts,
            ignore_index=True
        )

        assert (
            inner_pred[
                "fruit_id"
            ].nunique()
            ==
            32
        )

        score_fb = (
            fruit_balanced_mae(
                inner_pred
            )
        )

        score_mae = float(
            mean_absolute_error(
                inner_pred["y_true"],
                inner_pred["y_pred"]
            )
        )

        row = {
            "camera_type": camera,
            "outer_fold": int(outer_fold),
            "model": model_name,
            "preprocessing": preprocessing,
            "params_json": json.dumps(
                params,
                sort_keys=True
            ),
            "config_id": config_key,
            "inner_fruit_balanced_mae": score_fb,
            "inner_mae": score_mae,
            "failed": False,
            "failure": None,
        }

        search_rows.append(row)

        rank_tuple = (
            score_fb,
            score_mae,
            config_key
        )

        if (
            best is None
            or
            rank_tuple < best["rank"]
        ):

            best = {
                "rank": rank_tuple,
                "preprocessing": preprocessing,
                "params": params,
                "inner_fruit_balanced_mae": score_fb,
                "inner_mae": score_mae,
                "config_id": config_key,
            }

    if best is None:
        raise RuntimeError(
            f"No hubo configuración válida: "
            f"{camera} / outer {outer_fold} / {model_name}"
        )

    return (
        best,
        pd.DataFrame(
            search_rows
        )
    )


# 13. Evaluación externa OOF con checkpoints

Cada combinación `camera × outer_fold × model` se guarda inmediatamente.

Si Colab se desconecta, vuelva a ejecutar el notebook con `RESUME=True`.


In [ ]:
def safe_token(text):
    return (
        str(text)
        .replace("/", "_")
        .replace(" ", "_")
    )


def checkpoint_paths(
    camera,
    outer_fold,
    model_name
):

    stem = (
        f"{RUN_SIGNATURE}__"
        f"{safe_token(camera)}__"
        f"outer{int(outer_fold)}__"
        f"{safe_token(model_name)}"
    )

    return {
        "pred":
            CHECKPOINT_DIR /
            f"{stem}__pred.csv",

        "selected":
            CHECKPOINT_DIR /
            f"{stem}__selected.json",

        "search":
            CHECKPOINT_DIR /
            f"{stem}__inner_search.csv",
    }


def checkpoint_complete(paths):

    return (
        paths["pred"].exists()
        and
        paths["selected"].exists()
    )


In [ ]:
MODEL_NAMES = [
    "DummyMedian",
    "Ridge",
    "PLSR",
    "SVR",
]

all_prediction_parts = []
all_selected_rows = []
all_search_parts = []

start_time = time.time()

for camera in [
    "VIS",
    "VIS_COR"
]:

    df_camera = spectral_data[
        camera
    ].copy()

    wl_cols = [
        c for c in df_camera.columns
        if c.startswith("wl_")
    ]

    wavelength_grid = wavelengths[
        camera
    ]

    for outer_fold in range(
        OUTER_FOLDS
    ):

        test_fruits = set(
            outer_splits.loc[
                outer_splits[
                    "outer_fold"
                ]
                ==
                outer_fold,
                "fruit_id"
            ].astype(int)
        )

        development_fruits = (
            set(
                fruit_stats[
                    "fruit_id"
                ].astype(int)
            )
            -
            test_fruits
        )

        train_df = df_camera[
            df_camera[
                "fruit_id"
            ].isin(
                development_fruits
            )
        ].copy()

        test_df = df_camera[
            df_camera[
                "fruit_id"
            ].isin(
                test_fruits
            )
        ].copy()

        assert (
            set(
                train_df[
                    "fruit_id"
                ].astype(int)
            )
            &
            set(
                test_df[
                    "fruit_id"
                ].astype(int)
            )
            ==
            set()
        )

        inner_assignment = inner_splits[
            inner_splits[
                "outer_fold"
            ]
            ==
            outer_fold
        ][
            [
                "fruit_id",
                "inner_fold"
            ]
        ].copy()

        assert (
            set(
                inner_assignment[
                    "fruit_id"
                ].astype(int)
            )
            ==
            development_fruits
        )

        for model_name in MODEL_NAMES:

            paths = checkpoint_paths(
                camera,
                outer_fold,
                model_name
            )

            if (
                RESUME
                and
                checkpoint_complete(
                    paths
                )
            ):

                pred_df = pd.read_csv(
                    paths["pred"]
                )

                selected = json.loads(
                    paths["selected"]
                    .read_text(
                        encoding="utf-8"
                    )
                )

                all_prediction_parts.append(
                    pred_df
                )

                all_selected_rows.append(
                    selected
                )

                if paths["search"].exists():
                    all_search_parts.append(
                        pd.read_csv(
                            paths["search"]
                        )
                    )

                print(
                    f"↪ checkpoint: "
                    f"{camera} / outer {outer_fold} / {model_name}"
                )

                continue

            print(
                "\n",
                "=" * 72,
                sep=""
            )

            print(
                f"{camera} | outer fold {outer_fold} | {model_name}"
            )

            print(
                "=" * 72
            )

            X_train_raw = train_df[
                wl_cols
            ].to_numpy(
                dtype=float
            )

            X_test_raw = test_df[
                wl_cols
            ].to_numpy(
                dtype=float
            )

            y_train = train_df[
                "storage_days_model"
            ].to_numpy(
                dtype=float
            )

            if model_name == "DummyMedian":

                train_median = float(
                    np.median(
                        y_train
                    )
                )

                pred = np.repeat(
                    train_median,
                    len(test_df)
                )

                selected = {
                    "camera_type":
                        camera,

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "model":
                        model_name,

                    "preprocessing":
                        "none",

                    "params_json":
                        json.dumps(
                            {
                                "train_median":
                                    train_median
                            },
                            sort_keys=True
                        ),

                    "inner_fruit_balanced_mae":
                        None,

                    "inner_mae":
                        None,
                }

            else:

                best, search_df = (
                    inner_select_configuration(
                        camera=camera,
                        model_name=model_name,
                        outer_fold=outer_fold,
                        data_df=train_df,
                        wavelength_grid=wavelength_grid,
                        inner_assignment=inner_assignment,
                    )
                )

                preprocessing = best[
                    "preprocessing"
                ]

                params = best[
                    "params"
                ]

                X_train = preprocess_spectra(
                    X_train_raw,
                    wavelength_grid,
                    preprocessing
                )

                X_test = preprocess_spectra(
                    X_test_raw,
                    wavelength_grid,
                    preprocessing
                )

                pred, _ = fit_predict_estimator(
                    model_name,
                    params,
                    X_train,
                    y_train,
                    X_test
                )

                selected = {
                    "camera_type":
                        camera,

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "model":
                        model_name,

                    "preprocessing":
                        preprocessing,

                    "params_json":
                        json.dumps(
                            params,
                            sort_keys=True
                        ),

                    "inner_fruit_balanced_mae":
                        float(
                            best[
                                "inner_fruit_balanced_mae"
                            ]
                        ),

                    "inner_mae":
                        float(
                            best[
                                "inner_mae"
                            ]
                        ),
                }

                search_df.to_csv(
                    paths["search"],
                    index=False
                )

                all_search_parts.append(
                    search_df
                )

            pred_df = pd.DataFrame({
                "camera_type":
                    camera,

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "model":
                    model_name,

                "fruit_id":
                    test_df[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    test_df[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "day_code":
                    test_df[
                        "day_code"
                    ].astype(
                        str
                    ).to_numpy(),

                "y_true":
                    test_df[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    np.asarray(
                        pred,
                        dtype=float
                    ),
            })

            # Anti-leakage final por combinación
            assert (
                pred_df[
                    "fruit_id"
                ].nunique()
                ==
                8
            )

            assert set(
                pred_df[
                    "fruit_id"
                ].astype(int)
            ) == test_fruits

            pred_df.to_csv(
                paths["pred"],
                index=False
            )

            paths[
                "selected"
            ].write_text(
                json.dumps(
                    selected,
                    indent=2,
                    ensure_ascii=False
                ),
                encoding="utf-8"
            )

            all_prediction_parts.append(
                pred_df
            )

            all_selected_rows.append(
                selected
            )

oof_predictions = pd.concat(
    all_prediction_parts,
    ignore_index=True
)

selected_configs = pd.DataFrame(
    all_selected_rows
)

if all_search_parts:

    inner_search_results = pd.concat(
        all_search_parts,
        ignore_index=True
    )

else:

    inner_search_results = pd.DataFrame()

elapsed = time.time() - start_time

print(
    "\n✅ Evaluación terminada/reconstruida.",
    f"Tiempo de esta ejecución: {elapsed/60:.1f} min"
)


## 14. Auditoría de integridad OOF


In [ ]:
expected_models = set(
    MODEL_NAMES
)

for camera in [
    "VIS",
    "VIS_COR"
]:

    for model_name in MODEL_NAMES:

        g = oof_predictions[
            (
                oof_predictions[
                    "camera_type"
                ]
                ==
                camera
            )
            &
            (
                oof_predictions[
                    "model"
                ]
                ==
                model_name
            )
        ]

        assert len(g) == 199
        assert g["fruit_id"].nunique() == 40

        assert (
            g[
                [
                    "fruit_id",
                    "storage_days"
                ]
            ]
            .duplicated()
            .sum()
            ==
            0
        )

        # Cada fruto debe haber sido predicho solo en su fold externo.
        per_fruit_fold = (
            g.groupby(
                "fruit_id"
            )["outer_fold"]
            .nunique()
        )

        assert (
            per_fruit_fold == 1
        ).all()

assert (
    set(
        oof_predictions[
            "model"
        ]
    )
    ==
    expected_models
)

print(
    "✅ OOF íntegro: 199 predicciones por cámara y modelo, "
    "40 frutos, sin duplicación fruit-day."
)


## 15. Métricas por fold externo


In [ ]:
fold_metric_rows = []

for (
    camera,
    model_name,
    outer_fold
), g in oof_predictions.groupby(
    [
        "camera_type",
        "model",
        "outer_fold"
    ]
):

    metrics = metric_bundle(
        g
    )

    metrics.update({
        "camera_type": camera,
        "model": model_name,
        "outer_fold": int(outer_fold),
    })

    fold_metric_rows.append(
        metrics
    )

outer_fold_metrics = pd.DataFrame(
    fold_metric_rows
)

display(
    outer_fold_metrics.sort_values(
        [
            "camera_type",
            "model",
            "outer_fold"
        ]
    )
)

outer_fold_metrics.to_csv(
    TABLES /
    "05_outer_fold_metrics.csv",
    index=False
)


## 16. Métricas OOF agregadas


In [ ]:
aggregate_rows = []

for (
    camera,
    model_name
), g in oof_predictions.groupby(
    [
        "camera_type",
        "model"
    ]
):

    metrics = metric_bundle(
        g
    )

    fold_subset = outer_fold_metrics[
        (
            outer_fold_metrics[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            outer_fold_metrics[
                "model"
            ]
            ==
            model_name
        )
    ]

    metrics.update({
        "camera_type": camera,
        "model": model_name,
        "fold_fruit_balanced_mae_mean":
            float(
                fold_subset[
                    "fruit_balanced_mae"
                ].mean()
            ),
        "fold_fruit_balanced_mae_std":
            float(
                fold_subset[
                    "fruit_balanced_mae"
                ].std(
                    ddof=1
                )
            ),
    })

    aggregate_rows.append(
        metrics
    )

aggregate_metrics = (
    pd.DataFrame(
        aggregate_rows
    )
    .sort_values(
        [
            "camera_type",
            "fruit_balanced_mae"
        ]
    )
    .reset_index(drop=True)
)

display(
    aggregate_metrics
)

aggregate_metrics.to_csv(
    TABLES /
    "06_oof_aggregate_metrics.csv",
    index=False
)


# 17. Bootstrap por fruto para MAE fruit-balanced

Se remuestrean **40 frutos con reemplazo**, conservando todas las observaciones longitudinales de cada fruto.

El intervalo es descriptivo para la métrica OOF y respeta la unidad biológica.


In [ ]:
def bootstrap_fruit_balanced_mae(
    prediction_df,
    n_boot=2000,
    seed=20260818
):

    fruit_mae = (
        prediction_df
        .assign(
            abs_error=lambda x:
                np.abs(
                    x["y_true"]
                    -
                    x["y_pred"]
                )
        )
        .groupby(
            "fruit_id"
        )["abs_error"]
        .mean()
    )

    values = fruit_mae.to_numpy(
        dtype=float
    )

    rng_local = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot,
        dtype=float
    )

    for b in range(n_boot):

        sampled = rng_local.choice(
            values,
            size=len(values),
            replace=True
        )

        boot[b] = np.mean(
            sampled
        )

    return {
        "estimate":
            float(
                np.mean(
                    values
                )
            ),

        "ci95_low":
            float(
                np.percentile(
                    boot,
                    2.5
                )
            ),

        "ci95_high":
            float(
                np.percentile(
                    boot,
                    97.5
                )
            ),
    }


bootstrap_rows = []

for idx, (
    (
        camera,
        model_name
    ),
    g
) in enumerate(
    oof_predictions.groupby(
        [
            "camera_type",
            "model"
        ]
    )
):

    result = (
        bootstrap_fruit_balanced_mae(
            g,
            n_boot=BOOTSTRAP_REPS,
            seed=(
                RANDOM_SEED
                +
                idx * 100
            )
        )
    )

    bootstrap_rows.append({
        "camera_type": camera,
        "model": model_name,
        "fruit_balanced_mae":
            result[
                "estimate"
            ],
        "ci95_low":
            result[
                "ci95_low"
            ],
        "ci95_high":
            result[
                "ci95_high"
            ],
        "bootstrap_reps":
            BOOTSTRAP_REPS,
    })

bootstrap_ci = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_ci.sort_values(
        [
            "camera_type",
            "fruit_balanced_mae"
        ]
    )
)

bootstrap_ci.to_csv(
    TABLES /
    "07_fruit_bootstrap_mae_ci.csv",
    index=False
)


# 18. Configuraciones elegidas dentro de cada outer fold

La estabilidad de la selección es informativa:

- si el mismo preprocesamiento aparece repetidamente, existe estabilidad;
- si los hiperparámetros cambian mucho, la muestra puede ser insuficiente para identificar una configuración única.


In [ ]:
selected_configs = (
    selected_configs
    .sort_values(
        [
            "camera_type",
            "model",
            "outer_fold"
        ]
    )
    .reset_index(drop=True)
)

display(
    selected_configs
)

selected_configs.to_csv(
    TABLES /
    "08_selected_configs_by_outer_fold.csv",
    index=False
)

selection_stability = (
    selected_configs[
        selected_configs[
            "model"
        ]
        !=
        "DummyMedian"
    ]
    .groupby(
        [
            "camera_type",
            "model",
            "preprocessing",
            "params_json"
        ]
    )
    .size()
    .reset_index(
        name="n_outer_folds_selected"
    )
    .sort_values(
        [
            "camera_type",
            "model",
            "n_outer_folds_selected"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)

display(
    selection_stability
)

selection_stability.to_csv(
    TABLES /
    "09_selection_stability.csv",
    index=False
)

if not inner_search_results.empty:

    inner_search_results.to_csv(
        TABLES /
        "10_inner_search_results.csv",
        index=False
    )


# 19. Elegir el mejor baseline por cámara para visualización

Esta elección se usa **solo para resumir y graficar el resultado OOF**.

No constituye todavía un modelo final de despliegue ni una validación externa.


In [ ]:
trainable_metrics = aggregate_metrics[
    aggregate_metrics[
        "model"
    ]
    !=
    "DummyMedian"
].copy()

winner_rows = []

for camera, g in trainable_metrics.groupby(
    "camera_type"
):

    best = (
        g.sort_values(
            [
                "fruit_balanced_mae",
                "mae",
                "rmse"
            ]
        )
        .iloc[0]
    )

    winner_rows.append({
        "camera_type":
            camera,

        "best_baseline_model":
            best[
                "model"
            ],

        "fruit_balanced_mae":
            float(
                best[
                    "fruit_balanced_mae"
                ]
            ),

        "mae":
            float(
                best[
                    "mae"
                ]
            ),

        "rmse":
            float(
                best[
                    "rmse"
                ]
            ),

        "r2":
            float(
                best[
                    "r2"
                ]
            ),

        "within_1_day_accuracy":
            float(
                best[
                    "within_1_day_accuracy"
                ]
            ),
    })

camera_winners = pd.DataFrame(
    winner_rows
)

display(
    camera_winners
)

camera_winners.to_csv(
    TABLES /
    "11_camera_baseline_winners.csv",
    index=False
)


## 20. Figura — MAE fruit-balanced por modelo


In [ ]:
for camera, g in aggregate_metrics.groupby(
    "camera_type"
):

    order = [
        model
        for model in MODEL_NAMES
        if model in set(
            g["model"]
        )
    ]

    plot_df = (
        g.set_index(
            "model"
        )
        .loc[
            order
        ]
        .reset_index()
    )

    fig, ax = plt.subplots(
        figsize=(7.5, 4.8)
    )

    ax.bar(
        plot_df["model"],
        plot_df[
            "fruit_balanced_mae"
        ]
    )

    ax.set_xlabel(
        "Modelo"
    )

    ax.set_ylabel(
        "MAE fruit-balanced (días)"
    )

    ax.set_title(
        f"{camera}: desempeño OOF fruit-disjoint"
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"02_model_comparison_fruit_balanced_mae_{camera}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


## 21. Figura — predicho vs. real para el mejor baseline OOF


In [ ]:
for _, winner in camera_winners.iterrows():

    camera = winner[
        "camera_type"
    ]

    model_name = winner[
        "best_baseline_model"
    ]

    g = oof_predictions[
        (
            oof_predictions[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            oof_predictions[
                "model"
            ]
            ==
            model_name
        )
    ].copy()

    fig, ax = plt.subplots(
        figsize=(6.2, 5.5)
    )

    scatter = ax.scatter(
        g["y_true"],
        g["y_pred"],
        c=g["fruit_id"],
        s=34,
        alpha=0.8
    )

    lim_low = min(
        -0.5,
        float(
            g[
                [
                    "y_true",
                    "y_pred"
                ]
            ].min().min()
        )
        -
        0.25
    )

    lim_high = max(
        8.5,
        float(
            g[
                [
                    "y_true",
                    "y_pred"
                ]
            ].max().max()
        )
        +
        0.25
    )

    ax.plot(
        [
            lim_low,
            lim_high
        ],
        [
            lim_low,
            lim_high
        ],
        linestyle="--"
    )

    ax.set_xlim(
        lim_low,
        lim_high
    )

    ax.set_ylim(
        lim_low,
        lim_high
    )

    ax.set_xlabel(
        "Storage days reales"
    )

    ax.set_ylabel(
        "Storage days predichos OOF"
    )

    ax.set_title(
        f"{camera} — {model_name}\n"
        "OOF fruit-disjoint"
    )

    cb = fig.colorbar(
        scatter,
        ax=ax
    )

    cb.set_label(
        "Fruit ID"
    )

    ax.grid(
        alpha=0.15
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"03_predicted_vs_true_{camera}_{model_name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


## 22. Error absoluto por día — mejor baseline


In [ ]:
day_error_rows = []

for _, winner in camera_winners.iterrows():

    camera = winner[
        "camera_type"
    ]

    model_name = winner[
        "best_baseline_model"
    ]

    g = oof_predictions[
        (
            oof_predictions[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            oof_predictions[
                "model"
            ]
            ==
            model_name
        )
    ].copy()

    g["abs_error"] = np.abs(
        g["y_true"]
        -
        g["y_pred"]
    )

    summary_day = (
        g.groupby(
            "y_true"
        )["abs_error"]
        .agg(
            [
                "count",
                "mean",
                "median",
                "std"
            ]
        )
        .reset_index()
        .rename(
            columns={
                "y_true":
                    "storage_days"
            }
        )
    )

    summary_day[
        "camera_type"
    ] = camera

    summary_day[
        "model"
    ] = model_name

    day_error_rows.append(
        summary_day
    )

    fig, ax = plt.subplots(
        figsize=(8, 4.8)
    )

    ax.plot(
        summary_day[
            "storage_days"
        ],
        summary_day[
            "mean"
        ],
        marker="o",
        label="MAE"
    )

    ax.plot(
        summary_day[
            "storage_days"
        ],
        summary_day[
            "median"
        ],
        marker="s",
        label="Mediana AE"
    )

    ax.set_xlabel(
        "Storage days"
    )

    ax.set_ylabel(
        "Error absoluto (días)"
    )

    ax.set_title(
        f"{camera} — {model_name}: error por día"
    )

    ax.legend()

    ax.grid(
        alpha=0.2
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"04_error_by_storage_day_{camera}_{model_name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

day_error_summary = pd.concat(
    day_error_rows,
    ignore_index=True
)

display(
    day_error_summary
)

day_error_summary.to_csv(
    TABLES /
    "12_best_model_error_by_storage_day.csv",
    index=False
)


# 23. Error por fruto y relación con día terminal

Esta tabla permite detectar si los frutos con seguimiento más largo son sistemáticamente más difíciles o más fáciles.


In [ ]:
per_fruit_error_rows = []

terminal_map = (
    fruit_stats.set_index(
        "fruit_id"
    )["terminal_day"]
    .to_dict()
)

for _, winner in camera_winners.iterrows():

    camera = winner[
        "camera_type"
    ]

    model_name = winner[
        "best_baseline_model"
    ]

    g = oof_predictions[
        (
            oof_predictions[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            oof_predictions[
                "model"
            ]
            ==
            model_name
        )
    ].copy()

    g["abs_error"] = np.abs(
        g["y_true"]
        -
        g["y_pred"]
    )

    per_fruit = (
        g.groupby(
            "fruit_id"
        )["abs_error"]
        .agg(
            [
                "count",
                "mean",
                "median",
                "max"
            ]
        )
        .reset_index()
        .rename(
            columns={
                "count":
                    "n_fruit_days",

                "mean":
                    "fruit_mae",

                "median":
                    "fruit_median_ae",

                "max":
                    "fruit_max_ae",
            }
        )
    )

    per_fruit[
        "terminal_day"
    ] = per_fruit[
        "fruit_id"
    ].map(
        terminal_map
    )

    per_fruit[
        "camera_type"
    ] = camera

    per_fruit[
        "model"
    ] = model_name

    per_fruit_error_rows.append(
        per_fruit
    )

    fig, ax = plt.subplots(
        figsize=(7, 5)
    )

    ax.scatter(
        per_fruit[
            "terminal_day"
        ],
        per_fruit[
            "fruit_mae"
        ],
        s=42,
        alpha=0.8
    )

    ax.set_xlabel(
        "Día terminal del fruto"
    )

    ax.set_ylabel(
        "MAE del fruto (días)"
    )

    ax.set_title(
        f"{camera} — {model_name}: error por fruto"
    )

    ax.grid(
        alpha=0.2
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"05_per_fruit_mae_vs_terminal_day_{camera}_{model_name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

per_fruit_errors = pd.concat(
    per_fruit_error_rows,
    ignore_index=True
)

display(
    per_fruit_errors
    .sort_values(
        [
            "camera_type",
            "fruit_mae"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(20)
)

per_fruit_errors.to_csv(
    TABLES /
    "13_best_model_per_fruit_errors.csv",
    index=False
)


# 24. Comparación con DummyMedian

Se cuantifica la mejora del mejor baseline de cada cámara respecto a una predicción nula basada únicamente en la mediana de `y` del training de cada fold externo.


In [ ]:
dummy_comparison_rows = []

for _, winner in camera_winners.iterrows():

    camera = winner[
        "camera_type"
    ]

    best_model = winner[
        "best_baseline_model"
    ]

    best_row = aggregate_metrics[
        (
            aggregate_metrics[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            aggregate_metrics[
                "model"
            ]
            ==
            best_model
        )
    ].iloc[0]

    dummy_row = aggregate_metrics[
        (
            aggregate_metrics[
                "camera_type"
            ]
            ==
            camera
        )
        &
        (
            aggregate_metrics[
                "model"
            ]
            ==
            "DummyMedian"
        )
    ].iloc[0]

    improvement = (
        dummy_row[
            "fruit_balanced_mae"
        ]
        -
        best_row[
            "fruit_balanced_mae"
        ]
    )

    relative = (
        improvement
        /
        dummy_row[
            "fruit_balanced_mae"
        ]
        if dummy_row[
            "fruit_balanced_mae"
        ] > 0
        else np.nan
    )

    dummy_comparison_rows.append({
        "camera_type":
            camera,

        "best_model":
            best_model,

        "dummy_fruit_balanced_mae":
            float(
                dummy_row[
                    "fruit_balanced_mae"
                ]
            ),

        "best_fruit_balanced_mae":
            float(
                best_row[
                    "fruit_balanced_mae"
                ]
            ),

        "absolute_improvement_days":
            float(
                improvement
            ),

        "relative_improvement":
            float(
                relative
            ),
    })

dummy_comparison = pd.DataFrame(
    dummy_comparison_rows
)

display(
    dummy_comparison
)

dummy_comparison.to_csv(
    TABLES /
    "14_best_vs_dummy_comparison.csv",
    index=False
)


# 25. Guardar predicciones OOF y tablas centrales


In [ ]:
OOF_PATH = (
    RESULTS /
    "kaki_oof_predictions.csv"
)

SELECTED_PATH = (
    RESULTS /
    "kaki_selected_configs.csv"
)

METRICS_PATH = (
    RESULTS /
    "kaki_oof_aggregate_metrics.csv"
)

oof_predictions.to_csv(
    OOF_PATH,
    index=False
)

selected_configs.to_csv(
    SELECTED_PATH,
    index=False
)

aggregate_metrics.to_csv(
    METRICS_PATH,
    index=False
)

print("OOF:", OOF_PATH)
print("Configs:", SELECTED_PATH)
print("Metrics:", METRICS_PATH)


# 26. Resumen científico automático

El resumen separa explícitamente:

- desempeño OOF fruit-disjoint;
- selección exploratoria del mejor baseline;
- limitaciones que **no** resuelve este diseño.


In [ ]:
def json_safe(obj):

    if isinstance(obj, dict):
        return {
            str(k): json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [
            json_safe(v)
            for v in obj
        ]

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return (
            None
            if np.isnan(obj)
            else float(obj)
        )

    if isinstance(obj, np.bool_):
        return bool(obj)

    if isinstance(obj, np.ndarray):
        return json_safe(
            obj.tolist()
        )

    return obj


winner_payload = {}

for _, row in camera_winners.iterrows():

    camera = row[
        "camera_type"
    ]

    winner_payload[
        camera
    ] = {
        "model":
            row[
                "best_baseline_model"
            ],

        "fruit_balanced_mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),

        "mae":
            float(
                row[
                    "mae"
                ]
            ),

        "rmse":
            float(
                row[
                    "rmse"
                ]
            ),

        "r2":
            float(
                row[
                    "r2"
                ]
            ),

        "within_1_day_accuracy":
            float(
                row[
                    "within_1_day_accuracy"
                ]
            ),
    }


summary = {
    "project":
        PROJECT_NAME,

    "notebook":
        NOTEBOOK_ID,

    "run_signature":
        RUN_SIGNATURE,

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "validation_design": {
        "outer_folds":
            OUTER_FOLDS,

        "outer_test_fruits_per_fold":
            8,

        "inner_folds":
            INNER_FOLDS,

        "grouping_unit":
            "physical fruit_id",

        "outer_split_path":
            str(
                OUTER_SPLIT_PATH
            ),

        "inner_split_path":
            str(
                INNER_SPLIT_PATH
            ),

        "selection_metric":
            "fruit_balanced_mae",

        "leakage_audit_passed":
            True,
    },

    "data": {
        "physical_fruits":
            40,

        "fruit_days_per_camera":
            199,

        "storage_days":
            sorted(
                [
                    int(x)
                    for x in base[
                        "storage_days_model"
                    ].unique()
                ]
            ),

        "cameras":
            {
                "VIS":
                    int(
                        len(
                            wavelengths[
                                "VIS"
                            ]
                        )
                    ),

                "VIS_COR":
                    int(
                        len(
                            wavelengths[
                                "VIS_COR"
                            ]
                        )
                    ),
            },
    },

    "best_exploratory_baselines":
        winner_payload,

    "methodological_limits": [
        (
            "Fruit-disjoint validation prevents identity leakage "
            "but does not remove storage-day/acquisition-session confounding."
        ),
        (
            "The longitudinal panel is unbalanced because fruits leave "
            "the experiment at destructive endpoints."
        ),
        (
            "The OOF comparison is internal to one experimental campaign "
            "and is not equivalent to external batch/campaign validation."
        ),
        (
            "The best OOF baseline is exploratory and should not yet be "
            "described as a deployment-ready model."
        ),
    ],
}

SUMMARY_PATH = (
    RESULTS /
    "kaki_baseline_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    )
)


# 27. Generar notas para revisión

Estas notas indican qué debe revisarse antes de avanzar a modelos más complejos.


In [ ]:
review_lines = [
    "KAKI — BASELINE REVIEW",
    "",
    f"Run signature: {RUN_SIGNATURE}",
    "",
    "Validation:",
    "- 5 outer folds frozen, strictly fruit-disjoint.",
    "- 8 physical fruits in each outer test fold.",
    "- 4 inner fruit-disjoint folds for model/preprocessing selection.",
    "- Selection metric: fruit-balanced MAE.",
    "",
    "Models:",
    "- DummyMedian",
    "- Ridge",
    "- PLSR",
    "- SVR-RBF",
    "",
    "Preprocessing candidates:",
    "- raw",
    "- SNV",
    "- SG1 Savitzky-Golay first derivative",
    "",
    "Hard methodological limitations:",
    "- storage day remains linked to acquisition session.",
    "- one campaign only; no external campaign validation.",
    "- destructive endpoints produce an unbalanced longitudinal panel.",
    "",
    "Do not train CNN/HSI models before reviewing the OOF package.",
    "",
]

for _, row in camera_winners.iterrows():

    review_lines.extend([
        (
            f"{row['camera_type']} exploratory winner: "
            f"{row['best_baseline_model']}"
        ),
        (
            "  fruit-balanced MAE: "
            f"{row['fruit_balanced_mae']:.4f} days"
        ),
        (
            "  MAE: "
            f"{row['mae']:.4f} days"
        ),
        (
            "  RMSE: "
            f"{row['rmse']:.4f} days"
        ),
        (
            "  R2: "
            f"{row['r2']:.4f}"
        ),
        (
            "  ±1 day accuracy: "
            f"{row['within_1_day_accuracy']:.4f}"
        ),
        "",
    ])

REVIEW_NOTES_PATH = (
    RESULTS /
    "BASELINE_REVIEW_NOTES.txt"
)

REVIEW_NOTES_PATH.write_text(
    "\n".join(
        review_lines
    )
    +
    "\n",
    encoding="utf-8"
)

print(
    REVIEW_NOTES_PATH.read_text()
)
